# Regional Open Data Cubes — DEAfrica + DEA

Two of the new endpoints — `deafrica` and `dea` — are Open Data Cube STAC catalogs over their
respective continents. Both are anonymous (no credentials needed), and both expose a Water
Observations from Space (WOfS) product that flags every clear water pixel detected by Landsat.
This notebook pulls a WOfS observation from each over a small bbox to show the endpoint
abstraction — same request shape, different region, same `anonymous` signer.

## The two endpoints in the catalog (no network)

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
for ep in ("deafrica", "dea"):
    e = cat.endpoints[ep]
    print(f"{ep:9} url={e.url}")
    print(f"          signer={e.signer}  region={e.region}")
print()
print("deafrica wofs default_assets:", cat.get_collection("deafrica/wofs_ls").default_assets)
print("dea      wofs default_assets:", cat.get_collection("dea/ga_ls_wo_3").default_assets)

## DEAfrica WOfS over Johannesburg (live)

WOfS classifies each pixel into a small set of water-state codes; the `water` asset is the
single-band classification we want.

In [ ]:
import tempfile

from earthlens.earthlens import EarthLens

africa = EarthLens(
    data_source="deafrica",
    start="2024-01-01", end="2024-12-31",
    dataset="deafrica/wofs_ls",
    variables=["water"],
    aoi=[28.0, -26.5, 28.5, -26.0],   # Johannesburg
    path=tempfile.mkdtemp(), max_items=1,
)
africa_paths = africa.download()
africa_paths

## DEA WOfS over Canberra (live, same shape)

The only thing that changes is `data_source`, `dataset`, and the bbox. The signer is still
`anonymous`, the product still emits a single `water` asset.

In [ ]:
australia = EarthLens(
    data_source="dea",                    # alias 'digital-earth-australia' works too
    start="2024-01-01", end="2024-12-31",
    dataset="dea/ga_ls_wo_3",
    variables=["water"],
    aoi=[149.0, -35.5, 149.5, -35.0],     # Canberra
    path=tempfile.mkdtemp(), max_items=1,
)
aus_paths = australia.download()
aus_paths

## Inspect what each endpoint returned

Both rasters are single-band `uint8` WOfS classifications, projected to each endpoint's native
UTM zone. Reading them through pyramids exposes the CRS and shape.

In [ ]:
from pyramids.dataset import Dataset

for label, paths in (("DEAfrica", africa_paths), ("DEA", aus_paths)):
    ds = Dataset.read_file(str(paths[0]))
    print(f"{label:8} bands={ds.band_count} epsg={ds.epsg} shape={ds.read_array().shape}")

## Where to go next

- Both catalogs ship ~10 curated collections — landsat / sentinel-2 ARD, fractional cover,
  GeoMedian annual composites, Copernicus DEM mirrors. `Catalog().get_collection(<key>)` lists them.
- Swap to `data_source="deafrica"` / `"dea"` with `dataset="deafrica/fc_ls"` or
  `"dea/ga_ls8c_nbart_gm_cyear_3"` for fractional cover / annual NBART GeoMedian.